# Assignment 3: Milestone I Natural Language Processing
## Task 2 and Task 3
#### Student Name: (Fill in)
#### Student ID: (Fill in)

Environment: Python 3 and Jupyter notebook

Libraries used:
- pandas, numpy
- scikit-learn (TfidfVectorizer, LogisticRegression)
- gensim (Word2Vec)
- scipy
- pathlib, re, collections

## Introduction
This notebook implements Task 2 (feature representation) and Task 3 (classification) for cosmetics and beauty reviews.

Task 2 outputs generated by this notebook:
- `count_vectors.txt` (sparse unigram count vectors using `vocab.txt`)
- `unweighted_vectors.txt` (unweighted average Word2Vec word vectors)
- `weighted_vectors.txt` (TF-IDF weighted average Word2Vec word vectors)

Task 3 experiments use a simple model first (Logistic Regression) with 5-fold cross-validation to compare feature sets and answer both required questions.

## Importing Libraries

In [1]:
# !pip install -q --upgrade pip setuptools wheel
%pip install -q -r requirements.txt

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
from pathlib import Path
from collections import Counter
import re
import numpy as np
import pandas as pd
from scipy import sparse
from sklearn.model_selection import StratifiedKFold, cross_validate, train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    classification_report,
    confusion_matrix,
)
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

## Task 2. Generating Feature Representations for Cosmetics/Beauty Reviews

### 2.1 Count Vector Representation (Bag-of-Words)

The count vector representation encodes each review as a sparse vector where each dimension corresponds to a vocabulary word (from `vocab.txt`) and the value is the raw term frequency of that word in the `review_text` (excluding `review_title`).

- Vocabulary is loaded from `vocab.txt` (word → integer index, alphabetically sorted).
- For each review, tokenise the pre-processed `review_text` (space-separated tokens produced by Task 1) and count occurrences of vocabulary words.
- Only non-zero counts are stored (sparse format).
- Output format per line: `#review_index,word_integer_index:word_freq,...`
- Saved to `count_vectors.txt`.

In [3]:
PROCESSED_CSV = "processed.csv"
VOCAB = "vocab.txt"
COUNT_VECTOR = "count_vectors.txt"


def GenerateCountVector():
    # --- Load vocabulary (word -> integer index) ---
    vocab = {}
    with open(VOCAB, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            word, idx = line.rsplit(":", 1)
            vocab[word] = int(idx)

    # --- Load processed reviews ---
    df = pd.read_csv(PROCESSED_CSV)
    # review_text contains space-separated tokens produced by Task 1
    review_texts = df["review_text"].fillna("").astype(str).tolist()

    # --- Build and write sparse count vectors ---
    with open(COUNT_VECTOR, "w", encoding="utf-8") as out:
        for review_idx, text in enumerate(review_texts):
            tokens = text.split()
            # Count only tokens that exist in the vocabulary
            counts = Counter(token for token in tokens if token in vocab)
            # Sort by word integer index for a consistent ordering
            sparse_entries = sorted(
                (vocab[word], freq) for word, freq in counts.items()
            )
            sparse_str = ",".join(f"{idx}:{freq}" for idx, freq in sparse_entries)
            out.write(f"#{review_idx},{sparse_str}\n")

    print(f"Count vectors saved to '{COUNT_VECTOR}' ({len(review_texts)} reviews).")


# --- Run ---
GenerateCountVector()

Count vectors saved to 'count_vectors.txt' (61284 reviews).


### 2.2 Word Embedding Vectors (FastText — Unweighted & Weighted)

A pretrained **FastText** model (`fasttext-wiki-news-subwords-300`, 300-dimensional) is loaded via `gensim.downloader` and used to build two document-level vector representations for each review (`review_text` only):

- **Unweighted** — simple arithmetic mean of the FastText vectors of all valid tokens in the review.
- **Weighted** — TF-IDF weighted mean: each token's vector is scaled by its TF-IDF score (fitted across the full corpus) before averaging, giving more weight to informative terms.

Reviews with no recognisable tokens receive a zero vector. Both representations are saved in the same sparse-style line format:

```
#<review_index>,<v0>,<v1>,...,<v299>
```

Outputs: `unweighted_vectors.txt`, `weighted_vectors.txt`.

In [ ]:
import gensim.downloader as gensim_api

UNWEIGHTED_VECTOR = "unweighted_vectors.txt"
WEIGHTED_VECTOR = "weighted_vectors.txt"
FASTTEXT_MODEL_NAME = "fasttext-wiki-news-subwords-300"


def GenerateEmbeddingVectors():
    # --- Load pretrained FastText model ---
    print(
        f"Loading FastText model '{FASTTEXT_MODEL_NAME}' (downloads on first run ~960 MB)..."
    )
    fasttext_model = gensim_api.load(FASTTEXT_MODEL_NAME)
    vector_size = fasttext_model.vector_size
    print(f"Model loaded. Vector size: {vector_size}")

    # --- Load processed reviews ---
    df = pd.read_csv(PROCESSED_CSV)
    review_texts = df["review_text"].fillna("").astype(str).tolist()

    # --- Fit TF-IDF over the full corpus (for weighted representation) ---
    # tokenizer=str.split preserves the already-cleaned tokens from Task 1
    tfidf = TfidfVectorizer(tokenizer=str.split, lowercase=False, token_pattern=None)
    tfidf_matrix = tfidf.fit_transform(review_texts)
    tfidf_feature_names = tfidf.get_feature_names_out()
    tfidf_vocab = {word: idx for idx, word in enumerate(tfidf_feature_names)}

    # --- Generate and write vectors ---
    with open(UNWEIGHTED_VECTOR, "w", encoding="utf-8") as uw_out, open(
        WEIGHTED_VECTOR, "w", encoding="utf-8"
    ) as w_out:

        for review_idx, text in enumerate(review_texts):
            tokens = text.split()
            # Keep only tokens the FastText model knows
            valid_tokens = [t for t in tokens if t in fasttext_model]

            if valid_tokens:
                vectors = np.array([fasttext_model[t] for t in valid_tokens])

                # Unweighted: simple average of word vectors
                unweighted_vec = vectors.mean(axis=0)

                # Weighted: TF-IDF weighted average
                tfidf_row = tfidf_matrix[review_idx]
                weights = np.array(
                    [
                        tfidf_row[0, tfidf_vocab[t]] if t in tfidf_vocab else 0.0
                        for t in valid_tokens
                    ]
                )
                weight_sum = weights.sum()
                if weight_sum > 0:
                    weighted_vec = (vectors * weights[:, np.newaxis]).sum(
                        axis=0
                    ) / weight_sum
                else:
                    weighted_vec = unweighted_vec
            else:
                unweighted_vec = np.zeros(vector_size)
                weighted_vec = np.zeros(vector_size)

            uw_out.write(
                f"#{review_idx}," + ",".join(f"{v:.6f}" for v in unweighted_vec) + "\n"
            )
            w_out.write(
                f"#{review_idx}," + ",".join(f"{v:.6f}" for v in weighted_vec) + "\n"
            )

    print(
        f"Unweighted vectors saved to '{UNWEIGHTED_VECTOR}' ({len(review_texts)} reviews)."
    )
    print(
        f"Weighted vectors saved to '{WEIGHTED_VECTOR}' ({len(review_texts)} reviews)."
    )


# --- Run ---
GenerateEmbeddingVectors()

ImportError: cannot import name 'triu' from 'scipy.linalg.special_matrices' (d:\source\RMIT\master-of-ai-new\2026-semester-01\cosc3081-cosc3082-advanced-programming-for-data-science\.venv\Lib\site-packages\scipy\linalg\special_matrices.py)